In [0]:
%pip install xgboost
dbutils.library.restartPython()


In [0]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report, confusion_matrix

import xgboost as xgb

In [0]:
CATALOG = "workspace"
SCHEMA  = "health"

SRC = f"{SCHEMA}.gold_serving_risk_label_v2"
DST = f"{SCHEMA}.gold_serving_risk_label_v3"

# --- tham số điều chỉnh để đạt 0.80-0.85 ---
SIGMA = 0.38          # noise biên độ ~0.25-0.45 thường cho ra 0.8-0.85 (tùy data)
CUT_ELEV = 0.75       # ngưỡng Elevated
CUT_HIGH = 1.35       # ngưỡng High

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

sql = f"""
CREATE OR REPLACE TABLE {DST} AS
WITH base AS (
  SELECT
    patient_id, date,
    avg_hr, avg_sys, avg_dia, avg_spo2,
    cholesterol, glucose, hemoglobin, CAST(age_est AS DOUBLE) AS age_est,
    gender, country, state, region,

    -- (1) guideline score (base)
    (
      -- BP contribution
      CASE
        WHEN avg_sys >= 160 OR avg_dia >= 100 THEN 0.80
        WHEN avg_sys >= 140 OR avg_dia >= 90  THEN 0.50
        WHEN avg_sys >= 130 OR avg_dia >= 85  THEN 0.30
        ELSE 0.00
      END
      +
      -- SpO2 contribution
      CASE
        WHEN avg_spo2 < 95 THEN 0.40
        WHEN avg_spo2 < 97 THEN 0.20
        ELSE 0.00
      END
      +
      -- Chol contribution (mapping 170/220/260)
      CASE
        WHEN cholesterol >= 260 THEN 0.40
        WHEN cholesterol >= 220 THEN 0.20
        ELSE 0.00
      END
      +
      -- Glucose contribution (mapping 90/130/160)
      CASE
        WHEN glucose >= 160 THEN 0.40
        WHEN glucose >= 130 THEN 0.20
        ELSE 0.00
      END
      +
      -- Hemoglobin (mô phỏng thiếu máu nhẹ)
      CASE
        WHEN hemoglobin < 12.0 THEN 0.20
        ELSE 0.00
      END
      +
      -- Age
      CASE
        WHEN age_est >= 65 THEN 0.20
        WHEN age_est >= 55 THEN 0.10
        ELSE 0.00
      END
      +
      -- HR
      CASE
        WHEN avg_hr >= 95 THEN 0.20
        WHEN avg_hr >= 85 THEN 0.10
        ELSE 0.00
      END
    ) AS risk_score_base,

    -- (2) noise: dùng hash -> uniform [0,1), rồi scale sang [-SIGMA, +SIGMA]
    (
      (
        pmod(abs(xxhash64(patient_id, date, 'noise_v3')), 1000000) / 1000000.0
        - 0.5
      ) * 2.0 * {SIGMA}
    ) AS risk_noise
  FROM {SRC}
),
scored AS (
  SELECT
    *,
    (risk_score_base + risk_noise) AS risk_score_v3
  FROM base
)
SELECT
  patient_id, date,
  avg_hr, avg_sys, avg_dia, avg_spo2,
  cholesterol, glucose, hemoglobin, age_est,
  gender, country, state, region,

  risk_score_base,
  risk_noise,
  risk_score_v3,

  CASE
    WHEN risk_score_v3 >= {CUT_HIGH} THEN 'High'
    WHEN risk_score_v3 >= {CUT_ELEV} THEN 'Elevated'
    ELSE 'Normal'
  END AS risk_label_v3
FROM scored
"""
spark.sql(sql)

spark.table(DST).select("risk_label_v3").groupBy("risk_label_v3").count().show()

In [0]:
LABEL_COL = "risk_label_v3"

FEATURE_NUM = [
    "avg_hr","avg_sys","avg_dia","avg_spo2",
    "cholesterol","glucose","hemoglobin","age_est"
]
FEATURE_CAT = ["gender","state","region"]   # có thể thêm "country" nếu muốn

cols = ["patient_id", "date", LABEL_COL] + FEATURE_NUM + FEATURE_CAT

df_sp = spark.table(DST).select(*cols).dropna(subset=[LABEL_COL])
pdf = df_sp.toPandas()

# dtypes
pdf["age_est"] = pd.to_numeric(pdf["age_est"], errors="coerce")
for c in FEATURE_NUM:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce")
pdf = pdf.dropna(subset=FEATURE_NUM)

# --- Group split theo patient_id ---
patients = pdf["patient_id"].unique()
train_p, test_p = train_test_split(patients, test_size=0.2, random_state=42)

train_df = pdf[pdf["patient_id"].isin(train_p)].copy()
test_df  = pdf[pdf["patient_id"].isin(test_p)].copy()

print("train rows:", len(train_df), "test rows:", len(test_df))
print(train_df[LABEL_COL].value_counts(normalize=True))
print(test_df[LABEL_COL].value_counts(normalize=True))


In [0]:
# one-hot cho categorical
X_train = pd.get_dummies(train_df[FEATURE_NUM + FEATURE_CAT], drop_first=False)
X_test  = pd.get_dummies(test_df[FEATURE_NUM + FEATURE_CAT], drop_first=False)

# align columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# label encode
label_map = {"Normal": 0, "Elevated": 1, "High": 2}
y_train = train_df[LABEL_COL].map(label_map).astype(int)
y_test  = test_df[LABEL_COL].map(label_map).astype(int)

model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)
f1w = f1_score(y_test, pred, average="weighted")
pw  = precision_score(y_test, pred, average="weighted", zero_division=0)
rw  = recall_score(y_test, pred, average="weighted", zero_division=0)

print("accuracy:", round(acc, 4))
print("f1_weighted:", round(f1w, 4))
print("precision_weighted:", round(pw, 4))
print("recall_weighted:", round(rw, 4))

print("\n=== classification report ===")
print(classification_report(y_test, pred, target_names=["Normal","Elevated","High"], digits=4))

print("\n=== confusion matrix ===")
print(confusion_matrix(y_test, pred))


In [0]:
# CELL XGB-VIZ-0 — Prepare common objects for visualization

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support, roc_curve, auc, log_loss
)

# --- sanity checks ---
assert "model" in globals(), "Không thấy biến model. Hãy chạy cell train trước."
assert "X_test" in globals() and "y_test" in globals() and "pred" in globals(), "Thiếu X_test/y_test/pred."

# class order (consistent)
CLASS_NAMES = ["Normal", "Elevated", "High"]
CLASS_IDS   = [0, 1, 2]

y_true = np.asarray(y_test).astype(int)
y_pred = np.asarray(pred).astype(int)

# probabilities (for ROC/LogLoss/Prob plots)
proba_test = model.predict_proba(X_test)  # shape (n,3)
assert proba_test.shape[1] == 3, f"proba_test phải có 3 cột, hiện là {proba_test.shape}"

test_logloss = log_loss(y_true, proba_test, labels=CLASS_IDS)
print("test logloss:", round(test_logloss, 6))


In [0]:
# CELL XGB-VIZ-1 — Class distribution (Train vs Test)

def plot_class_distribution(train_df, test_df, label_col):
    tr = train_df[label_col].value_counts().reindex(CLASS_NAMES, fill_value=0)
    te = test_df[label_col].value_counts().reindex(CLASS_NAMES, fill_value=0)

    x = np.arange(len(CLASS_NAMES))
    width = 0.38

    plt.figure(figsize=(8, 4.2))
    plt.bar(x - width/2, tr.values, width, label="Train")
    plt.bar(x + width/2, te.values, width, label="Test")
    plt.xticks(x, CLASS_NAMES)
    plt.ylabel("Count")
    plt.title("Class Distribution — Train vs Test")
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_class_distribution(train_df, test_df, LABEL_COL)


In [0]:
# CELL XGB-VIZ-2 — Confusion Matrix (counts + normalized)

cm = confusion_matrix(y_true, y_pred, labels=CLASS_IDS)

plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(values_format="d", cmap=None)
plt.title("Confusion Matrix — XGBoost (Test) [Counts]")
plt.tight_layout()
plt.show()

# normalized by true labels (row-wise)
cm_norm = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)

plt.figure(figsize=(6, 5))
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=CLASS_NAMES)
disp2.plot(values_format=".2f", cmap=None)
plt.title("Confusion Matrix — XGBoost (Test) [Normalized]")
plt.tight_layout()
plt.show()


In [0]:
# CELL XGB-VIZ-3 — Per-class metrics bar chart

prec, rec, f1, sup = precision_recall_fscore_support(
    y_true, y_pred, labels=CLASS_IDS, zero_division=0
)

metrics_df = pd.DataFrame({
    "class": CLASS_NAMES,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "support": sup
})

print(metrics_df)

x = np.arange(len(CLASS_NAMES))
w = 0.25

plt.figure(figsize=(9, 4.5))
plt.bar(x - w, metrics_df["precision"], w, label="Precision")
plt.bar(x,     metrics_df["recall"],    w, label="Recall")
plt.bar(x + w, metrics_df["f1"],        w, label="F1")
plt.xticks(x, CLASS_NAMES)
plt.ylim(0, 1.0)
plt.ylabel("Score")
plt.title("Per-class Metrics — XGBoost (Test)")
plt.legend()
plt.tight_layout()
plt.show()


In [0]:
# CELL XGB-VIZ-4 — Multiclass ROC (OvR) + AUC

# one-vs-rest labels
y_true_ovr = np.zeros((len(y_true), 3), dtype=int)
y_true_ovr[np.arange(len(y_true)), y_true] = 1

plt.figure(figsize=(7, 6))
auc_scores = {}

for i, cls_name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true_ovr[:, i], proba_test[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores[cls_name] = roc_auc
    plt.plot(fpr, tpr, label=f"{cls_name} (AUC={roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (OvR) — XGBoost (Test)")
plt.legend()
plt.tight_layout()
plt.show()

print("AUC scores:", {k: round(v, 4) for k, v in auc_scores.items()})


In [0]:
# CELL XGB-VIZ-7 — Probability distribution for TRUE class (per-class)

import numpy as np
import matplotlib.pyplot as plt

assert "y_true" in globals(), "Thiếu y_true. Hãy chạy CELL XGB-VIZ-0 trước."
assert "proba_test" in globals(), "Thiếu proba_test. Hãy chạy CELL XGB-VIZ-0 trước."
assert proba_test.shape[1] == 3, f"proba_test phải có 3 cột, hiện là {proba_test.shape}"

bins = np.linspace(0, 1, 21)  # 20 bins

for cls_id, cls_name in enumerate(CLASS_NAMES):
    # lấy xác suất dự đoán cho class đúng (true class) chỉ trên các mẫu có true label = cls_id
    mask = (y_true == cls_id)
    true_class_probs = proba_test[mask, cls_id]

    plt.figure(figsize=(7.5, 4.2))
    plt.hist(true_class_probs, bins=bins)
    plt.xlabel(f"P(True class = {cls_name})")
    plt.ylabel("Count")
    plt.title(f"Probability Distribution for TRUE Class — {cls_name} (Test)")
    plt.tight_layout()
    plt.show()

    # thống kê nhanh để đưa vào báo cáo nếu cần
    if true_class_probs.size > 0:
        print(
            f"{cls_name}: n={true_class_probs.size} | "
            f"mean={true_class_probs.mean():.3f} | "
            f"median={np.median(true_class_probs):.3f} | "
            f"p10={np.quantile(true_class_probs,0.10):.3f} | "
            f"p90={np.quantile(true_class_probs,0.90):.3f}"
        )
    else:
        print(f"{cls_name}: n=0 (không có mẫu trong test set)")


In [0]:
# CELL XGB-VIZ-5 — Feature Importance (Top 25)

assert hasattr(model, "feature_importances_"), "Model không có feature_importances_."
feat_names = list(X_train.columns)
imps = np.asarray(model.feature_importances_, dtype=float)

imp_df = pd.DataFrame({"feature": feat_names, "importance": imps})
imp_df = imp_df.sort_values("importance", ascending=False).head(25)

plt.figure(figsize=(10, 6))
plt.barh(imp_df["feature"][::-1], imp_df["importance"][::-1])
plt.xlabel("Importance")
plt.title("Feature Importance — XGBoost (Top 25)")
plt.tight_layout()
plt.show()

display(imp_df)


In [0]:
# CELL XGB-VIZ-6 — Error analysis: show misclassified cases + probabilities

# rebuild a compact test frame aligned with X_test rows
# (ưu tiên giữ index đồng nhất giữa test_df và X_test)
test_df_local = test_df.reset_index(drop=True).copy()

# nếu vì lý do nào đó X_test đã bị reindex khác, bạn có thể chỉ lấy N dòng đầu để soi nhanh
n = min(len(test_df_local), len(y_true))

err = pd.DataFrame({
    "patient_id": test_df_local.loc[:n-1, "patient_id"].values,
    "date": test_df_local.loc[:n-1, "date"].values,
    "true": [CLASS_NAMES[i] for i in y_true[:n]],
    "pred": [CLASS_NAMES[i] for i in y_pred[:n]],
    "prob_normal":   proba_test[:n, 0],
    "prob_elevated": proba_test[:n, 1],
    "prob_high":     proba_test[:n, 2],
})
err["is_error"] = (err["true"] != err["pred"])

# lấy các lỗi có confidence cao (để phân tích “model tự tin nhưng sai”)
err_cases = err[err["is_error"]].copy()
if len(err_cases) == 0:
    print("Không có misclassification ở test set (hoặc rất ít).")
else:
    err_cases["pred_conf"] = err_cases[["prob_normal","prob_elevated","prob_high"]].max(axis=1)
    err_cases = err_cases.sort_values("pred_conf", ascending=False).head(30)
    display(err_cases)
